In [25]:
import numpy as np
from importlib import reload
import matplotlib.pyplot as plt
%matplotlib widget
import sys
from pathlib import Path
# from scipy.stats import t, truncnorm
import joblib
import pandas as pd
import json

# import re

In [3]:
module_path = Path('..').resolve()
if str(module_path) not in sys.path:
    sys.path.insert(0, str(module_path))

In [26]:
import libs.emulator_libs as elibs
import libs.param_libs as plibs
elibs = reload(elibs)
plibs = reload(plibs)
# import libs.custom_prospector_tools as cpt

In [7]:
myparams = {'zred': 0.1,
            'logmass': 10,
            'logzsol': 0.0,
            'nbins': 5,
            'logsfr_ratios': [0.0, 0.0, 0.0, 0.0],
            'dust2': 1.0,
            'dust_ratio': 1.0,
            'dust_index': -1.2,
            'duste_gamma': 0.01,
            'duste_umin': 1.0,
            'duste_qpah': 2.0,
            'add_neb_emission': False,
            'add_neb_continuum': False,
            'gas_logz': 0.0,
            'gas_logu': -2.0,
            'add_agn': False,
            'fagn': 0.0001,
            'agn_tau': 5.0}

In [8]:
ntrain = int(1e5)
nvalid = int(1e4)
ntest = int(1e4)

seed_train = 123
seed_valid = 456
seed_test = 789

train_param_keys = [
    "zred",
    "logmass",
    "logzsol",
    "logsfr_ratios0",
    "logsfr_ratios1",
    "logsfr_ratios2",
    "logsfr_ratios3",
    # "logsfr_ratios4",
    # "logsfr_ratios5",
    # "logsfr_ratios6",
    # "logsfr_ratios7",
    "dust2",
    "dust_index",
    "duste_qpah",
]

prior_dicts = {
    "zred": {
        "bounds": [0.0, 3.0],
        "prior": {
            "dist": "loguniform_1pz"
        }
    },

    "logmass": {
        "bounds": [7.5, 13.5],
        "prior": {
            "dist": "uniform"
        }
    },

    "logzsol": {
        "bounds": [-2.0, 0.2],
        "prior": {
            "dist": "uniform"
        }
    },

    "logsfr_ratios": {
        "bounds": [-5.0, 5.0],
        "prior": {
            "dist": "student_t",
            "df": 2,
            "loc": 0.0,
            "scale": 3.0,
        }
    },

    "dust2": {
        "bounds": [0.0, 4.0],
        "prior": {
            "dist": "truncnorm",
            "loc": 0.3,
            "scale": 3.0,
        }
    },

    "dust_index": {
        "bounds": [-1.2, 0.4],
        "prior": {
            "dist": "uniform",
        }
    },

    "duste_qpah": {
        "bounds": [0.0, 7.0],
        "prior": {
            "dist": "truncnorm",
            "loc": 2.0,
            "scale": 4.0,
        }
    },

    # TODO other parameters
}

In [9]:
default_params = plibs.get_default_params(myparams, train_param_keys)
default_params


{'nbins': 5,
 'dust_ratio': 1.0,
 'duste_gamma': 0.01,
 'duste_umin': 1.0,
 'add_neb_emission': False,
 'add_neb_continuum': False,
 'gas_logz': 0.0,
 'gas_logu': -2.0,
 'add_agn': False,
 'fagn': 0.0001,
 'agn_tau': 5.0}

In [10]:
rand_vals = plibs.generate_random_values(prior_dicts=prior_dicts, train_param_keys=train_param_keys, nsamples=ntrain, rng=np.random.default_rng(seed_train))

In [11]:
rand_vals

array([[ 1.57523435,  7.6932124 , -1.5427232 , ...,  1.94068765,
         0.21732353,  3.88262528],
       [ 0.07746577,  9.79546954, -0.15602987, ...,  3.83867227,
        -0.47614282,  6.0206162 ],
       [ 0.35728129, 13.20864779, -1.23070661, ...,  0.47186026,
        -1.18241351,  3.84867268],
       ...,
       [ 0.98830028, 12.28860498, -1.3633243 , ...,  1.97965626,
        -0.01378948,  3.3952685 ],
       [ 0.01801655,  9.36259571, -1.45122425, ...,  0.90380281,
        -0.55393694,  1.62770987],
       [ 1.02274263,  9.974734  , -1.81070961, ...,  2.42369157,
         0.09534215,  0.21856216]], shape=(100000, 10))

In [27]:
all_myparams = plibs.rand_vals_to_all_params(rand_vals, train_param_keys, default_params)

In [29]:
all_myparams[0]

{'nbins': 5,
 'dust_ratio': 1.0,
 'duste_gamma': 0.01,
 'duste_umin': 1.0,
 'add_neb_emission': False,
 'add_neb_continuum': False,
 'gas_logz': 0.0,
 'gas_logu': -2.0,
 'add_agn': False,
 'fagn': 0.0001,
 'agn_tau': 5.0,
 'logsfr_ratios': [np.float64(0.6966207503074507),
  np.float64(-3.6300931146563844),
  np.float64(-0.7708176928677539),
  np.float64(-2.628209175541109)],
 'zred': np.float64(1.57523435164106),
 'logmass': np.float64(7.693212396544195),
 'logzsol': np.float64(-1.5427231963070145),
 'dust2': np.float64(1.9406876523766219),
 'dust_index': np.float64(0.21732353271004404),
 'duste_qpah': np.float64(3.8826252820374454)}

In [30]:
pwd

'/Users/brianwang76/SPHEREx/pwang55/prospector_emulator/notebooks'